# 🌟 Diabetes Progression Prediction: A Beginner's Guide

Welcome! In this notebook, we will build a **Multiple Linear Regression** model from scratch using Python and NumPy. 

Our goal is to predict the progression of diabetes in patients based on various health metrics. We will break down each part of the code to understand how it works.

## 1. Import Libraries

First, we need to import the necessary libraries.
*   **NumPy (`np`)**: Used for efficient numerical operations and matrix math.
*   **Pandas (`pd`)**: Used for loading and handling the dataset.
*   **Matplotlib (`plt`)**: Used for creating visualizations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

## 2. Load and Split the Dataset

We load our data from `assets/diabetes.csv`. 

To properly evaluate our model, we split the data into two sets:
*   **Training Set (80%)**: Used to train the model.
*   **Test Set (20%)**: Used to test how well the model generalizes to new data.

In [ ]:
# Load dataset
df = pd.read_csv('assets/diabetes.csv')

X = df.drop('Y', axis=1).values  # Features
y = df[['Y']].to_numpy()         # Target
m = len(y)

# 2) Train / Test split
test_ratio = 0.2
indices = np.random.permutation(m)

test_size = int(m * test_ratio)
test_idx  = indices[:test_size]
train_idx = indices[test_size:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Training samples: {len(y_train)}")
print(f"Test samples: {len(y_test)}")

## 3. Data Standardization

We use **Z-score Standardization** to scale our features. This involves subtracting the mean ($\mu$) and dividing by the standard deviation ($\sigma$).

$$ X_{scaled} = \frac{X - \mu}{\sigma} $$

**Important**: We calculate $\mu$ and $\sigma$ using only the **Training** set, and then apply those same values to scale the **Test** set.

In [ ]:
# Feature scaling (Standardization)
X_mean = X_train.mean(axis=0)
X_std  = X_train.std(axis=0)
X_std[X_std == 0] = 1.0  # Prevent division by zero

X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled  = (X_test  - X_mean) / X_std

## 4. Add Intercept Term

We add a column of **ones** to our feature matrices to account for the intercept term ($\theta_0$).

In [ ]:
X_train_scaled = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled])
X_test_scaled  = np.hstack([np.ones((X_test_scaled.shape[0], 1)),  X_test_scaled])

print(f"New shape of X_train: {X_train_scaled.shape}")

## 5. Hyperparameters & Convergence

We set our learning parameters. We also define a **convergence threshold** (`epsilon`). If the cost function stops decreasing significantly, we stop training early to save time.

In [ ]:
alpha = 0.01
max_iters = 10000
epsilon = 1e-3  # Convergence threshold

## 6. Initialize Theta

Initialize the parameter vector $\theta$ with zeros.

In [ ]:
theta = np.zeros((X_train_scaled.shape[1], 1))

## 7. Gradient Descent with Convergence Check

We iterate to minimize the cost function. We added a check: if `abs(prev_cost - cost) < epsilon`, we consider the model converged.

In [ ]:
def compute_cost(X, y, theta):
    m = len(y)
    predictions = X @ theta
    error = predictions - y
    cost = (1/(2*m)) * np.sum(error ** 2)
    return cost

cost_history = []
prev_cost = float('inf')

for i in range(max_iters):
    predictions = X_train_scaled @ theta
    error = predictions - y_train
    gradient = (1/len(y_train)) * (X_train_scaled.T @ error)
    theta -= alpha * gradient

    cost = compute_cost(X_train_scaled, y_train, theta)
    cost_history.append(cost)

    # Convergence check
    if abs(prev_cost - cost) < epsilon:
        print(f"Converged at iteration {i+1} with cost {cost:.4f}")
        break

    prev_cost = cost
else:
    print(f"Reached max iterations {max_iters}")

## 8. Final Results and Visualization

We evaluate the model on the **Test Set** to see how well it performs on unseen data.

In [ ]:
# Final predictions on Test Set
y_test_pred = X_test_scaled @ theta

test_mse = compute_cost(X_test_scaled, y_test, theta) * 2
print(f"Test MSE: {test_mse:.4f}")

# Plot Test Performance
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_test_pred, alpha=0.6, color='blue', label='Test Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'k--', lw=2, label='Perfect Prediction')
plt.xlabel("Actual Disease Progression (Test)")
plt.ylabel("Predicted Disease Progression (Test)")
plt.title("Test Set Performance (Standardized)")
plt.legend()
plt.grid(True)
plt.show()